# ResNet and ConvNeXtV2 Results Summary

This is a **results reproduction and presentation notebook**. It summarizes Joshua's CNN-based model work for the CSC3109 project from saved artifacts rather than training models inline:

- ResNet18 frozen feature extractor.
- ResNet18 frozen feature extractor with data augmentation.
- ResNet18 last-block fine-tuning.
- ConvNeXtV2 Tiny local experiment artifacts, when available.

The notebook reads existing JSON metrics and figures. It does not train models.

Canonical ResNet source paths after the owner-folder cleanup:

- Models: `src.models.resnet.frozen`, `src.models.resnet.finetune`
- Training: `src.training.resnet.frozen`, `src.training.resnet.frozen_augmented`, `src.training.resnet.finetune_last_block`
- Evaluation: `src.evaluation.resnet.evaluate_finetune`

The older root-level module names remain compatibility wrappers for existing commands and reports.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "reports").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing reports/ and notebooks/.")

PROJECT_ROOT = find_project_root()

def load_json(relative_path: str):
    path = PROJECT_ROOT / relative_path
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def metric_value(metrics: dict, *names: str):
    for name in names:
        if name in metrics:
            return metrics[name]
    return None

def support_count(metrics: dict):
    report = metrics.get("classification_report", {})
    weighted = report.get("weighted avg", {}) if isinstance(report, dict) else {}
    return weighted.get("support")

def metric_row(name: str, family: str, strategy: str, artifact: str, notes: str):
    metrics = load_json(artifact)
    if metrics is None:
        return {
            "model": name,
            "family": family,
            "strategy": strategy,
            "accuracy": None,
            "precision_macro": None,
            "recall_macro": None,
            "f1_macro": None,
            "best_epoch": None,
            "support": None,
            "artifact": artifact,
            "status": "missing",
            "notes": notes,
        }
    return {
        "model": name,
        "family": family,
        "strategy": strategy,
        "accuracy": metric_value(metrics, "accuracy", "best_val_accuracy"),
        "precision_macro": metric_value(metrics, "precision_macro", "macro_precision"),
        "recall_macro": metric_value(metrics, "recall_macro", "macro_recall"),
        "f1_macro": metric_value(metrics, "f1_macro", "macro_f1"),
        "best_epoch": metric_value(metrics, "best_epoch", "epoch"),
        "support": support_count(metrics),
        "artifact": artifact,
        "status": "loaded",
        "notes": notes,
    }

pd.set_option("display.max_colwidth", 120)
PROJECT_ROOT

## ResNet18 Runs

These are the controlled ResNet runs from the original manifest split. The main comparison is frozen transfer learning versus light fine-tuning.

Canonical commands for reproducing the three main ResNet runs are:

```bash
python -m src.training.resnet.frozen
python -m src.training.resnet.frozen_augmented --epochs 10 --batch-size 32
python -m src.training.resnet.finetune_last_block --epochs 10 --batch-size 32
```

In [ ]:
resnet_rows = [
    metric_row(
        "ResNet18 frozen",
        "ResNet18",
        "Frozen feature extractor, no augmentation",
        "reports/tables/resnet18_frozen_metrics.json",
        "Run 1 baseline.",
    ),
    metric_row(
        "ResNet18 frozen augmented",
        "ResNet18",
        "Frozen feature extractor, training-only augmentation",
        "reports/tables/resnet18_frozen_augmented_metrics.json",
        "Run 2 augmentation comparison.",
    ),
    metric_row(
        "ResNet18 fine-tuned last block",
        "ResNet18",
        "Fine-tune layer4 + fc, training-only augmentation",
        "reports/tables/resnet18_finetune_last_block_metrics.json",
        "Run 3 transfer learning versus fine-tuning comparison.",
    ),
    metric_row(
        "ResNet18 strict split seed 42",
        "ResNet18",
        "Fine-tune layer4 + fc, strict contiguous validation block",
        "reports/tables/resnet18_finetune_last_block_strict_seed42_metrics.json",
        "Strict split repeat using seed 42.",
    ),
    metric_row(
        "ResNet18 strict split seed 123",
        "ResNet18",
        "Fine-tune layer4 + fc, strict contiguous validation block",
        "reports/tables/resnet18_finetune_last_block_strict_seed123_metrics.json",
        "Strict split repeat using seed 123.",
    ),
    metric_row(
        "ResNet18 strict split seed 999",
        "ResNet18",
        "Fine-tune layer4 + fc, strict contiguous validation block",
        "reports/tables/resnet18_finetune_last_block_strict_seed999_metrics.json",
        "Strict split repeat using seed 999.",
    ),
]

resnet_df = pd.DataFrame(resnet_rows)
resnet_df[["model", "strategy", "accuracy", "precision_macro", "recall_macro", "f1_macro", "best_epoch", "status", "notes"]]

In [ ]:
plot_df = resnet_df.dropna(subset=["accuracy", "f1_macro"])
if not plot_df.empty:
    ax = plot_df.set_index("model")[["accuracy", "f1_macro"]].plot(kind="bar", figsize=(9, 4), ylim=(0.95, 1.005))
    ax.set_title("ResNet18 Transfer Learning vs Fine-Tuning")
    ax.set_ylabel("Score")
    ax.legend(loc="lower right")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
else:
    display(Markdown("No ResNet metrics were found to plot."))

## ConvNeXtV2 Tiny Runs

ConvNeXtV2 Tiny is the modern ConvNet baseline in `src.models.timm_classifier` under the alias `convnextv2-tiny`. These artifacts live under `model/`, which is ignored by Git, so this notebook marks them as missing when they are not present locally.

In [ ]:
convnext_rows = [
    metric_row(
        "ConvNeXtV2 Tiny linear probe",
        "ConvNeXtV2",
        "Frozen backbone / classifier-only timm run",
        "model/convnextv2_tiny_fcmae_ft_in1k_linear_probe/best_tune_metrics.json",
        "Local ignored artifact. Useful if support is 560 from the original manifest-style split.",
    ),
    metric_row(
        "ConvNeXtV2 Tiny linear probe seed123",
        "ConvNeXtV2",
        "Frozen backbone / classifier-only timm run, seed 123",
        "model/convnextv2_tiny_fcmae_ft_in1k_linear_probe_seed123/best_tune_metrics.json",
        "Local ignored artifact for seed comparison.",
    ),
    metric_row(
        "ConvNeXtV2 Tiny fine-tune",
        "ConvNeXtV2",
        "Full fine-tune timm run",
        "model/convnextv2_tiny_fcmae_ft_in1k_finetune/best_tune_metrics.json",
        "Treat as diagnostic if support is very small.",
    ),
]

convnext_df = pd.DataFrame(convnext_rows)
convnext_df[["model", "strategy", "accuracy", "precision_macro", "recall_macro", "f1_macro", "best_epoch", "support", "status", "notes"]]

## Saved ResNet Figures

In [ ]:
figure_paths = [
    ("ResNet18 frozen augmented confusion matrix", "reports/figures/resnet18_frozen_augmented_confusion_matrix.png"),
    ("ResNet18 frozen augmented training curves", "reports/figures/resnet18_frozen_augmented_training_curves.png"),
    ("ResNet18 fine-tuned last block confusion matrix", "reports/figures/resnet18_finetune_last_block_confusion_matrix.png"),
    ("ResNet18 fine-tuned last block training curves", "reports/figures/resnet18_finetune_last_block_training_curves.png"),
]

for title, relative_path in figure_paths:
    path = PROJECT_ROOT / relative_path
    display(Markdown(f"### {title}"))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Missing figure: `{relative_path}`"))

## Short Interpretation

- Frozen ResNet18 is the clean transfer-learning baseline: pretrained CNN features are reused and only the final classifier is trained.
- ResNet18 last-block fine-tuning adapts the strongest high-level ResNet features by training `layer4` and `fc` with separate learning rates.
- ConvNeXtV2 Tiny is a stronger modern ConvNet comparison point. Use the linear-probe result as reportable only when the local metric artifact has the expected validation support.
- The ConvNeXtV2 fine-tune artifact should be treated as diagnostic if its support is only a few samples.